# Single-chain model validation

Compare simulated RNA secondary-structure contacts against ViennaRNA MFE
and the radius of gyration reference. One point represents one **independent
trajectory**. All measurements come from the pipeline's last 10% summaries.

**Input:** `analysis_results/<single-chain-sweep>/replicates.csv`
and `summary.csv`. Rg is calculated during the single-chain pairing pass;
this notebook never reopens a GSD trajectory.

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Works when Jupyter starts from the repository or analysis/notebooks/.
here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


## Load the single-chain validation sweep

Set the sweep name to the actual input folder you analyzed. It is deliberately
not guessed or hardcoded to a thesis figure number.

In [ ]:
SWEEP = "SET_SINGLE_CHAIN_SWEEP_NAME"
replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
print(f"{len(replicates)} trajectories; {len(summary)} conditions")
replicates[["E_att", "angle_k_ppp", "mfe_matches", "extra_pairs", "precision", "recall", "Rg_nm"]].head()

## Precision versus recall by chain stiffness

In [ ]:
from src.plots.figures import plot_precision_recall
fig, axes = panels_by(
    replicates,
    by="angle_k_ppp",
    plot_fn=partial(plot_precision_recall, style=STYLE),
    style=STYLE,
    ncols=3,
    title="MFE reference: precision and recall",
)
plt.show()

## Native/MFE-matching and additional base-pair contacts

In [ ]:
from src.plots.figures import plot_metric_series
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
one_stiffness = sorted(summary["angle_k_ppp"].unique())[0]
chosen = filter_rows(summary, angle_k_ppp=one_stiffness)
for ax, metric in zip(axes, ("mfe_matches", "extra_pairs")):
    plot_metric_series(chosen, ax, metric=metric, style=STYLE, band="std")
fig.suptitle(f"At stiffness k = {one_stiffness:g}; means ± SD across replicates")
plt.show()

### Full stiffness comparison for MFE-matching and additional pairs

Each panel plots a separate calibrated backbone stiffness.

In [ ]:
for metric in ("mfe_matches", "extra_pairs"):
    fig, axes = panels_by(
        summary, by="angle_k_ppp",
        plot_fn=partial(plot_metric_series, metric=metric, style=STYLE, band="std"),
        style=STYLE, title=STYLE.label(metric),
    )
    plt.show()

## Radius of gyration

`Rg_nm` comes from unwrapped particle coordinates saved in the validation
pairing NPZ. The dashed line marks the configured 1.47 nm reference used in
the thesis; it is a structural comparison, not a fitted result.

In [ ]:
fig, axes = panels_by(
    summary,
    by="angle_k_ppp",
    plot_fn=partial(plot_metric_series, metric="Rg_nm", style=STYLE,
                    band="p10_p90"),
    style=STYLE,
    title="Single-chain radius of gyration (mean and 10th–90th percentiles)",
)
for ax in axes.flat:
    if ax.get_visible():
        ax.axhline(1.47, ls="--", lw=1, color="#666", label="Reference")
        ax.legend(frameon=False)
plt.show()

## Takeaway

Discuss how stiffness and attraction influence secondary-structure specificity
and chain compaction. These figures are generated from the same versioned
analysis outputs, without copying plotting implementations into notebook cells.